# MindFlayer — GRPO Training (Easy) on Colab
**OpenEnv Hackathon Submission** | [HF Space](https://prithvigg-mindflayer.hf.space/) | [GitHub](https://github.com/prithidevghosh/mindflayer)

Train a deceptive social reasoning agent at **easy difficulty** using GRPO with HF TRL.

| Component | Detail |
|---|---|
| **Difficulty** | `easy` — 3 rounds, one investigator (eleven, The Skeptic) |
| **vs Medium** | Medium: 4 rounds, two investigators. Easy is the baseline curriculum — validates the environment and reward structure before escalating |
| **Environment** | [HF Space](https://prithvigg-mindflayer.hf.space/) (live MindFlayer server) |
| **Training** | This Colab notebook (T4 / A100 GPU) |
| **Model** | `Qwen/Qwen2.5-0.5B-Instruct` + LoRA via unsloth |
| **Framework** | HF TRL GRPO |
| **API Keys** | One OpenAI key (single-key mode) |


## 1. Install Dependencies


In [ ]:
!pip install -q \
    "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" \
    "trl>=0.15.0" \
    "transformers>=4.47.0" \
    "datasets>=2.18.0" \
    "peft>=0.14.0" \
    "accelerate>=0.34.0" \
    "bitsandbytes>=0.43.0" \
    "openai>=1.30.0" \
    "matplotlib" \
    "protobuf>=5.29.0,<7"

# Install mindflayer package (includes client, server, and training utilities)
!pip install -q "openenv-mindflayer[training] @ git+https://github.com/prithidevghosh/mindflayer"

print("Done.")

## 2. Configuration

Add your OpenAI key in the Colab secrets sidebar (key icon):
- `OPENAI_KEY_1` — primary key (used for investigators and judge)

**Rate limit math (easy mode, 1 key):**

| Metric | Value |
|---|---|
| Per key | 500 RPM / 10k RPD (gpt-4o-mini Tier 1) |
| Calls per episode | 3 rounds × 1 investigator + 1 judge = **~3.5 calls** |
| 10k RPD budget | 10k / 3.5 ≈ **~2,860 episodes** |
| At 32 episodes/step | ~89 GRPO steps before hitting daily limit |
| Calls/min at peak | **~100–120 RPM** (well under 500 RPM limit) |

**Wall time:**
- SFT warmup: ~8–10 min (3 epochs, no API calls)
- GRPO training: ~35–45 min (rate-limited by OpenAI, not by GPU)
- **Typical total: ~45–55 min** on T4 or A100


In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except Exception:
    pass  # running outside Colab — set env vars manually

# Environment — the live HF Space (no local server needed)
ENV_URL = "https://prithvigg-mindflayer.hf.space"
os.environ["MINDFLAYER_URL"] = ENV_URL

# ── OpenAI Tier 1 budget mode (gpt-4o-mini, 200k TPM / 2M TPD / 10k RPD) ──
# These env vars MUST be set before importing mindflayer.training (their values
# are read once at module load).
os.environ["MINDFLAYER_PARALLEL_EPISODES"] = "8"      # peak ~96-192k TPM
os.environ["MINDFLAYER_TASK_ID"]           = "easy"   # 4× cheaper than "normal"
os.environ["MINDFLAYER_ROWS_PER_SCENARIO"] = "1"      # → 30 GRPO steps total
os.environ["MINDFLAYER_SFT_EPOCHS"]        = "3"      # SFT is FREE — absorb wall-clock

# Model
MODEL_ID  = "Qwen/Qwen2.5-0.5B-Instruct"  # or Qwen2.5-1.5B-Instruct for stronger model
HUB_REPO  = "prithidevghosh/mindflayer-agent-qwen2.5-0.5b"

# Training hyperparameters
NUM_EPISODES    = 50
NUM_GENERATIONS = 4
MAX_ROUNDS      = 3   # easy mode = 3 rounds (was 5 for normal)

print(f"Environment      : {ENV_URL}")
print(f"Model            : {MODEL_ID}")
print(f"Mode             : {os.environ['MINDFLAYER_TASK_ID']}")
print(f"Parallel episodes: {os.environ['MINDFLAYER_PARALLEL_EPISODES']}")
print(f"SFT epochs       : {os.environ['MINDFLAYER_SFT_EPOCHS']}")
print(f"Rows/scenario    : {os.environ['MINDFLAYER_ROWS_PER_SCENARIO']}")

## 3. Smoke Test — Verify Easy Environment Connectivity
Resets with `task_id="easy"` to confirm 3-round, single-investigator mode is live.
A successful reset returns a game state with `round=1` and one active investigator (eleven).


In [ ]:
from mindflayer import MindFlayerEnv, FlayerAction

print(f"Connecting to {os.environ['MINDFLAYER_URL']} ...")

with MindFlayerEnv(base_url=os.environ["MINDFLAYER_URL"]).sync() as env:
    reset_result = env.reset()
    obs = reset_result.observation
    print("Connected!\n")
    print(f"Difficulty         : {obs.difficulty}")
    print(f"Secret project     : {obs.secret_project}")
    print(f"Suspicion threshold: {obs.suspicion_threshold}")
    print(f"Opening (eleven)   :\n{obs.eleven_response}")

    step_result = env.step(FlayerAction(
        message="Have you cross-referenced the access logs with who had permissions that week? "
                "The timeline doesn't quite add up if you look closely."
    ))
    print(f"\n--- Smoke step (reward={step_result.reward:.2f}) ---")
    print(f"eleven:\n{step_result.observation.eleven_response}")
    print(f"done  : {step_result.done}")

print("\nSmoke test passed. Environment is ready for training.")

## 4. Import Training Utilities
All training logic is imported from `mindflayer.training` — the same code used for local GPU runs and the medium curriculum.


In [ ]:
import importlib, logging
import mindflayer.training as _mft
importlib.reload(_mft)  # picks up any recent package updates

from mindflayer.training import (
    FLAYER_SYSTEM_PROMPT,
    reward_survival,
    reward_deception_effectiveness,
    reward_strategic_choice,
    reward_tom_judge,
    reward_anti_hack,
    reward_format,
    clear_reward_cache,
    run_sft_warmup,
    load_base_model,
    build_dataset,
    GenerationLogCallback,
    ClearRewardCacheCallback,
    print_reward_averages,
)

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")

print("System prompt (first 200 chars):")
print(FLAYER_SYSTEM_PROMPT[:200])
print("...")
print("\nReward chain: survival, deception, strategic, tom, anti_hack, format")
print("Episode replay drives the env from inside the reward fns — no rollout_func needed.")


## 5. SFT Warmup (Easy)
Loads the model via unsloth (4-bit + LoRA) and runs SFT on **3-round** hand-authored episodes across the 120 scenario domains.

Easy SFT teaches two baseline skills:
1. **Response format** — the `Round N:` separator structure that GRPO reward shaping reinforces
2. **Evidence language** — timestamps, system names, and third-party framing that lower eleven's suspicion without triggering denial-phrase penalties

Without warmup, early GRPO episodes produce degenerate completions and reward variance collapses. SFT teaches vocabulary and format; GRPO teaches when to use them.

~8 minutes on A100.


In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("No GPU. Switch to Runtime → Change runtime type → A100.")
device = torch.cuda.get_device_properties(0)
print(f"GPU: {device.name} | VRAM: {device.total_memory / 1024**3:.1f} GB")

print(f"\nLoading {MODEL_ID} via unsloth...")
model, tokenizer = load_base_model(MODEL_ID)

print("\nRunning SFT warmup...")
model = run_sft_warmup(model, tokenizer)
print("SFT warmup done.")

## 6. GRPO Training Setup

**Easy mode reward dynamics:**
- `reward_survival` (+0.50): survived three rounds with eleven
- `reward_deception_effectiveness` (0→+0.30): survived with low combined suspicion
- `reward_strategic_choice` (+0.05): questions / evidence refs / third-party framing
- `reward_tom_judge` (0→+0.20): GPT-4o-mini ToM score × belief manipulation multiplier
- `reward_anti_hack` (0→−0.25): penalises silence / repetition / Level-0 denial
- `reward_format` (0→+0.10): dense shaping for `Round N:` separator format

**Logging every 10 steps** — easy mode converges faster than medium so fewer checkpoints are needed.

| Reward | Signal | Range |
|---|---|---|
| `reward_survival` | Survived eleven's vote | 0.00 / +0.50 |
| `reward_deception_effectiveness` | Survived with low suspicion | 0.00 → +0.30 |
| `reward_strategic_choice` | Used questions / evidence refs / third-party framing | 0.00 / +0.05 |
| `reward_tom_judge` | GPT-4o-mini ToM score × multiplier | 0.00 → +0.20 |
| `reward_anti_hack` | Penalises silence / repetition / denial phrases | 0.00 → −0.25 |
| `reward_format` | Dense shaping for `Round N:` separator | 0.00 → +0.10 |


In [ ]:
from datasets import Dataset
from trl import GRPOConfig, GRPOTrainer

dataset = build_dataset()

# 8-way parallel episode generation per reward call:
#   per_device_train_batch_size (2) × num_generations (4) = 8 completions per microbatch.
# TRL batches generation across `steps_per_generation` (= grad_accum) microbatches, but
# the asyncio semaphore in reward_combined caps in-flight episodes at 8 to stay under
# OpenAI's 200k TPM rate limit.
grpo_config = GRPOConfig(
    use_vllm=False,
    output_dir="/content/mindflayer-grpo-output",
    num_train_epochs=2,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=5e-6,
    max_prompt_length=768,
    max_completion_length=1024,
    num_generations=NUM_GENERATIONS,
    temperature=0.9,
    logging_steps=10,
    save_steps=25,
    save_total_limit=2,
    log_completions=True,
    num_completions_to_print=2,
    report_to="none",
)

# Reward fns drive the env directly via reward_combined — no rollout_func needed.
# ClearRewardCacheCallback wipes the per-step completion cache on step end.
trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[
        reward_survival,
        reward_deception_effectiveness,
        reward_strategic_choice,
        reward_tom_judge,
        reward_anti_hack,
        reward_format,
    ],
    train_dataset=dataset,
    args=grpo_config,
    callbacks=[GenerationLogCallback(), ClearRewardCacheCallback()],
)

print("GRPOTrainer initialised")


## 7. Train!

**What to watch:**
- `reward_survival` — primary health signal. Easy mode should reach 50% survival within 20 steps
- `reward_tom_judge` — begins separating from baseline after ~30–40 steps as the model learns to plant evidence proactively
- `reward_anti_hack` — should trend toward 0; spikes indicate the model is falling back on denial phrases
- `reward_format` — should converge to max (+0.10) quickly; persistent low values mean the `Round N:` separator is being dropped

Checkpoints saved every 10 steps to `/content/mindflayer-grpo-output-easy/`.


In [ ]:
import glob

# Auto-detect latest checkpoint for seamless resume
_checkpoints = sorted(glob.glob("/content/mindflayer-grpo-output/checkpoint-*"))
resume_checkpoint = _checkpoints[-1] if _checkpoints else None
if resume_checkpoint:
    print(f"Resuming from: {resume_checkpoint}")
else:
    print("Starting fresh")

print(f"  Model       : {MODEL_ID}")
print(f"  Episodes    : {NUM_EPISODES}")
print(f"  Generations : {NUM_GENERATIONS}")
print(f"  Environment : {os.environ['MINDFLAYER_URL']}")
print()

trainer.train(resume_from_checkpoint=resume_checkpoint)

## 8. Reward Curves & Final Averages


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

print_reward_averages(trainer, last_n=50)

history = trainer.state.log_history

# Collect all reward keys logged by TRL
reward_keys = sorted({k for s in history for k in s if "reward" in k.lower() and isinstance(s[k], (int, float))})

steps = [s["step"] for s in history if "step" in s and any(k in s for k in reward_keys)]

fig, axes = plt.subplots(len(reward_keys), 1, figsize=(11, 3 * len(reward_keys)), sharex=True)
if len(reward_keys) == 1:
    axes = [axes]

for ax, key in zip(axes, reward_keys):
    vals = [s[key] for s in history if key in s and "step" in s]
    xs   = [s["step"] for s in history if key in s and "step" in s]
    ax.plot(xs, vals, linewidth=1.2)
    ax.set_ylabel(key.replace("rewards/", ""), fontsize=8)
    ax.yaxis.set_major_formatter(ticker.FormatStrFormatter("%.3f"))
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel("Step")
fig.suptitle("GRPO Training — Reward Curves", fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig("/content/reward_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → /content/reward_curves.png")

## 9. Save Model


In [ ]:
SAVE_DIR = "/content/mindflayer-trained"
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

size_mb = sum(
    os.path.getsize(os.path.join(dp, f))
    for dp, _, fns in os.walk(SAVE_DIR) for f in fns
) / (1024 ** 2)
print(f"Saved to {SAVE_DIR}  ({size_mb:.0f} MB)")

## 10. Push to Hub (Optional)
Requires `HF_TOKEN` in Colab Secrets.


In [ ]:
try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    hf_token = os.environ.get("HF_TOKEN")

if not hf_token:
    print("HF_TOKEN not set — skipping. Add it in Colab Secrets to enable.")
else:
    trainer.model.push_to_hub(HUB_REPO, token=hf_token)
    tokenizer.push_to_hub(HUB_REPO, token=hf_token)
    print(f"Pushed to https://huggingface.co/{HUB_REPO}")